In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, date
import ast

In [ ]:
def ensure_columns(df, fill_value=pd.NA):

    required_columns = ['json_name', 'column_name', 'path',
                        'list_path', 'subfield_path', 'var_type', 
                        'data_type', 'file_path', 'id', 'keepID']
    
    for col in required_columns:
        if col not in df.columns:
            df[col] = fill_value

    return df

In [ ]:
def combine_list(df):

    df['final_path'] = None

    for ix, row in df.iterrows():
        if row['platform'] == 'Tiktok':
            final_path =  row['path']
            
        else:
            file_list = row['file_path'].split('/')
            path_list = row['path']
        
            
            if isinstance(path_list, str):
                path_list = ast.literal_eval(path_list)

            if isinstance(path_list, list):
                final_path = file_list + path_list
                path = '/'.join(path_list)


        final_path = '/'.join(final_path)

        df.at[ix, 'final_path'] = final_path 
        df.at[ix, 'path'] = path       
    
    return df
    


In [ ]:

ROOT = '/home/rvissche/GIT/social-media-data-map'
root_dir = Path(f"{ROOT}/data/raw/annotated_merged_structures")
save_dir = f'{ROOT}/data/processed'
  

def create_dataset(root_dir, save_dir):

    dfs_list = []

    for csv_file in root_dir.glob("*.csv"):

        try:
            file_name =  csv_file.stem
            
            
            df = pd.read_csv(csv_file)
            

            if 'TT_' in str(csv_file):
                df['platform'] = 'Tiktok'
            if 'IG_' in str(csv_file):
                df['platform'] = 'Instagram'
            if 'FB_' in str(csv_file):
                df['platform'] = 'Facebook'
            if 'YT_' in str(csv_file):
                df['platform'] = 'Youtube'
            if 'X_' in str(csv_file):
                df['platform'] = 'Twitter'

            df = combine_list(df)
            
            col = df.pop('platform') 
            df.insert(1, 'platform', col) 

            df = ensure_columns(df)
            df = df.dropna(subset=['keepID'])
            
            dfs_list.append(df)
            print('FILE_NAME done: ', file_name)
        
        except Exception as e:
            print(f"Failed to load {csv_file}: {e}")


    dfs = pd.concat(dfs_list, ignore_index=True)




    print('DATASET CREATED at ', datetime.now())
    dfs.to_csv(f'{save_dir}/annotated_paths{date.today()}.csv')
    print('DATASET SAVED TO ', save_dir)

    #return dfs

create_dataset(root_dir, save_dir)

